# ⚙️ Adım 3 — Veri Ön İşleme & Hasta Bazında Veri Bölme

Bu notebook'ta iki kritik işi yapıyoruz:

**A) Hasta bazında train / validation / test bölme**  
**B) Ön işleme pipeline kurma**

Neden bu sıra? Çünkü ön işleme **sadece train verisine fit edilmeli**.  
Test verisindeki bilgi, eğitim sürecine sızmamalı — buna **data leakage** denir.

> **Finansal analoji:** Bir trading modeli kurarken gelecek verilerini kullanarak geçmişi tahmin etmek gibi. Sonuçlar harika görünür ama gerçek hayatta çalışmaz.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

# Veri yükle
normal = pd.read_csv('../data/normal_radiomics.csv')
papil  = pd.read_csv('../data/papilodem_radiomics.csv')
normal['label'] = 0
papil['label']  = 1
df = pd.concat([normal, papil], ignore_index=True)

feature_cols = [c for c in df.columns if c.startswith('Feature_')]
print(f'Toplam veri: {df.shape[0]} satır, {len(feature_cols)} özellik')
print(f'Unique hasta: {df["PatientIndex"].nunique()}')

---
## BÖLÜM A — Hasta Bazında Veri Bölme

### Neden satır bazında değil, hasta bazında bölüyoruz?

Her hastanın 14 satırı var. Eğer rastgele bölsek, aynı hastanın 10 satırı train'e, 4 satırı test'e düşebilir.  
Model bu hastayı zaten "tanıyor" — yani test performansı sahte olarak şişiyor.

**Flowchart'a göre bölme oranları:**
- Train: %70 hasta
- Validation: %10 hasta  
- Test: %20 hasta (final değerlendirmeye kadar hiç dokunulmaz)

**20 tekrar:** Her seferinde farklı rastgele bölme yapıp sonuçları ortalarız → daha güvenilir performans tahmini.

In [ ]:
def hasta_bazinda_bol(df, test_ratio=0.20, val_ratio=0.10, random_state=42):
    """
    Hastaları train / val / test olarak böler.
    Aynı hastanın satırları kesinlikle farklı setlere düşmez.
    Sınıf oranını korumak için stratified bölme yapılır.
    """
    rng = np.random.RandomState(random_state)
    
    # Her hasta için sınıf etiketi (bir hastanın tek etiketi var)
    hasta_etiket = df.groupby('PatientIndex')['label'].first()
    
    train_idx, val_idx, test_idx = [], [], []
    
    # Her sınıf için ayrı ayrı böl (stratified)
    for sinif in [0, 1]:
        hastalar = hasta_etiket[hasta_etiket == sinif].index.tolist()
        rng.shuffle(hastalar)
        
        n = len(hastalar)
        n_test = max(1, int(n * test_ratio))
        n_val  = max(1, int(n * val_ratio))
        
        test_idx  += hastalar[:n_test]
        val_idx   += hastalar[n_test:n_test + n_val]
        train_idx += hastalar[n_test + n_val:]
    
    train_df = df[df['PatientIndex'].isin(train_idx)].copy()
    val_df   = df[df['PatientIndex'].isin(val_idx)].copy()
    test_df  = df[df['PatientIndex'].isin(test_idx)].copy()
    
    return train_df, val_df, test_df


# Tek bir bölme örneği göster
train_df, val_df, test_df = hasta_bazinda_bol(df, random_state=42)

print('=== VERİ BÖLME SONUCU (tek örnek) ===')
for isim, veri in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    h  = veri['PatientIndex'].nunique()
    s  = len(veri)
    p  = veri['label'].mean() * 100
    print(f'{isim:12s}: {h:2d} hasta, {s:3d} satır, papilödem oranı: %{p:.0f}')

# Overlap kontrolü — hiçbir hasta iki grupta olmamalı
train_h = set(train_df['PatientIndex'])
val_h   = set(val_df['PatientIndex'])
test_h  = set(test_df['PatientIndex'])

print(f'\nOverlap kontrolü:')
print(f'  Train ∩ Val  : {len(train_h & val_h)} hasta (0 olmalı)')
print(f'  Train ∩ Test : {len(train_h & test_h)} hasta (0 olmalı)')
print(f'  Val ∩ Test   : {len(val_h & test_h)} hasta (0 olmalı)')

In [ ]:
# 20 farklı split oluştur ve sakla (outer loop)
N_SPLITS = 20
splits = []

for i in range(N_SPLITS):
    tr, va, te = hasta_bazinda_bol(df, random_state=i * 7 + 42)
    splits.append({
        'split_id'  : i,
        'train'     : tr,
        'val'       : va,
        'test'      : te
    })

# Özet tablo
ozet = pd.DataFrame([{
    'Split': s['split_id'],
    'Train hasta': s['train']['PatientIndex'].nunique(),
    'Val hasta'  : s['val']['PatientIndex'].nunique(),
    'Test hasta' : s['test']['PatientIndex'].nunique(),
    'Train satır': len(s['train']),
    'Test papilödem %': round(s['test']['label'].mean()*100, 1)
} for s in splits])

print(f'{N_SPLITS} outer split hazır. Örnek istatistikler:')
display(ozet.head(5))
print(f'\nOrtalama train hasta: {ozet["Train hasta"].mean():.1f}')
print(f'Ortalama test hasta : {ozet["Test hasta"].mean():.1f}')

---
## BÖLÜM B — Ön İşleme Pipeline

4 adım, sırayla uygulanacak:

| Adım | Yöntem | Neden? |
|------|--------|--------|
| 1 | Median Imputation | Eksik değerleri ortalama yerine medyanla doldur (aykırı değerlere dayanıklı) |
| 2 | Low-Variance Filtering | EDA'dan hatırlıyoruz: 327 özellik neredeyse sabit, bunlar modele gürültü katar |
| 3 | Korelasyon Eleme (>0.95) | 504 özellik birbirinin kopyası, 242'ye indiriyoruz |
| 4 | RobustScaler | Ölçekleri eşitle, aykırı değerlerden korunmak için medyan & IQR kullan |

> **Kritik kural:** Her adım SADECE train verisine `fit` edilir. Val ve test'e sadece `transform` uygulanır.

In [ ]:
class RadyomikOnIsleme:
    """
    Tüm ön işleme adımlarını sıralı uygulayan pipeline.
    fit() sadece train verisine çağrılır.
    transform() train, val ve test'e ayrı ayrı uygulanır.
    """
    
    def __init__(self, variance_threshold=0.01, correlation_threshold=0.95):
        self.variance_threshold    = variance_threshold
        self.correlation_threshold = correlation_threshold
        
        self.imputer       = SimpleImputer(strategy='median')
        self.var_selector  = VarianceThreshold(threshold=variance_threshold)
        self.scaler        = RobustScaler()
        
        self.selected_features_after_var  = None  # Varyans filtresi sonrası
        self.selected_features_after_corr = None  # Korelasyon filtresi sonrası
        self.high_corr_to_drop            = None
    
    def fit(self, X_train, feature_names):
        """Sadece train verisine fit et."""
        print(f'Başlangıç özellik sayısı: {X_train.shape[1]}')
        
        # 1. Median Imputation
        X = self.imputer.fit_transform(X_train)
        print(f'  [1] Median imputation ✓')
        
        # 2. Low-Variance Filtering
        self.var_selector.fit(X)
        mask_var = self.var_selector.get_support()
        self.selected_features_after_var = [f for f, m in zip(feature_names, mask_var) if m]
        X = X[:, mask_var]
        print(f'  [2] Varyans filtresi  ✓  → {X.shape[1]} özellik kaldı '
              f'({sum(~mask_var)} elendi)')
        
        # 3. Korelasyon bazlı eleme
        corr_df = pd.DataFrame(X, columns=self.selected_features_after_var)
        corr_matrix = corr_df.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.high_corr_to_drop = [
            col for col in upper.columns if any(upper[col] > self.correlation_threshold)
        ]
        self.selected_features_after_corr = [
            f for f in self.selected_features_after_var if f not in self.high_corr_to_drop
        ]
        keep_idx = [self.selected_features_after_var.index(f)
                    for f in self.selected_features_after_corr]
        X = X[:, keep_idx]
        print(f'  [3] Korelasyon eleme  ✓  → {X.shape[1]} özellik kaldı '
              f'({len(self.high_corr_to_drop)} elendi)')
        
        # 4. RobustScaler
        self.scaler.fit(X)
        print(f'  [4] RobustScaler      ✓')
        print(f'\nFinal özellik sayısı: {X.shape[1]}')
        
        return self
    
    def transform(self, X, feature_names):
        """Fit edilmiş pipeline'ı herhangi bir veri setine uygula."""
        # 1. Imputation
        X = self.imputer.transform(X)
        # 2. Varyans filtresi
        feat_df = pd.DataFrame(X, columns=feature_names)
        X = feat_df[self.selected_features_after_var].values
        # 3. Korelasyon eleme
        feat_df2 = pd.DataFrame(X, columns=self.selected_features_after_var)
        X = feat_df2[self.selected_features_after_corr].values
        # 4. Ölçekleme
        X = self.scaler.transform(X)
        return X
    
    def fit_transform(self, X_train, feature_names):
        self.fit(X_train, feature_names)
        return self.transform(X_train, feature_names)


print('RadyomikOnIsleme sınıfı tanımlandı ✓')

In [ ]:
# İlk split üzerinde test edelim
split0 = splits[0]

X_train = split0['train'][feature_cols].values
X_val   = split0['val'][feature_cols].values
X_test  = split0['test'][feature_cols].values

y_train = split0['train']['label'].values
y_val   = split0['val']['label'].values
y_test  = split0['test']['label'].values

print('=== ÖN İŞLEME — Split 0 ===' )
pipeline = RadyomikOnIsleme(variance_threshold=0.01, correlation_threshold=0.95)

# Sadece train'e fit et
X_train_proc = pipeline.fit_transform(X_train, feature_cols)

# Val ve test'e sadece transform
X_val_proc  = pipeline.transform(X_val,  feature_cols)
X_test_proc = pipeline.transform(X_test, feature_cols)

print(f'\nBoyutlar:')
print(f'  X_train : {X_train.shape} → {X_train_proc.shape}')
print(f'  X_val   : {X_val.shape}   → {X_val_proc.shape}')
print(f'  X_test  : {X_test.shape}  → {X_test_proc.shape}')

In [ ]:
# Ölçekleme başarılı mı? Train verisinin dağılımı kontrol
sample_feats = np.random.choice(range(X_train_proc.shape[1]), 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, idx in zip(axes.flatten(), sample_feats):
    ax.hist(X_train_proc[:, idx], bins=20, color='steelblue',
            alpha=0.7, edgecolor='white', label='Train')
    ax.hist(X_val_proc[:, idx],   bins=20, color='salmon',
            alpha=0.5, edgecolor='white', label='Val')
    ax.set_title(f'Özellik #{idx}', fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('RobustScaler Sonrası Özellik Dağılımları (Train vs Val)', fontsize=11)
plt.tight_layout()
plt.savefig('../figures/fig_olcekleme_sonrasi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Train ve Val dağılımları benzer görünmeli — çok farklıysa data leakage var demektir.')

---
## Özet & Sonraki Adım

In [ ]:
n_final = len(pipeline.selected_features_after_corr)

print('=' * 55)
print('ÖN İŞLEME ÖZET')
print('=' * 55)
print(f'Başlangıç özelliği              : 746')
print(f'Varyans filtresi sonrası        : {len(pipeline.selected_features_after_var)}')
print(f'Korelasyon eleme sonrası        : {n_final}')
print(f'Toplam indirim                  : %{(1 - n_final/746)*100:.0f}')
print(f'Outer split sayısı              : {N_SPLITS}')
print(f'Her split: train/val/test oranı : ~70/10/20')
print(f'Data leakage                    : YOK (fit sadece train)')
print('=' * 55)
print('\nSıradaki adım → 04_mrmr_ozellik_secimi.ipynb')
print(f'{n_final} özellikten MRMR ile en bilgilendirici k tanesi seçilecek.')